In [0]:
dbutils.widgets.text("catalog", "dev")
catalog = dbutils.widgets.get("catalog")

# Table creation

In [0]:
%sql
create table if not exists identifier(:catalog).bronze.bronze_customers;
create table if not exists identifier(:catalog).bronze.bronze_accounts;
create table if not exists identifier(:catalog).bronze.bronze_transactions;
create table if not exists identifier(:catalog).bronze.bronze_branches;

In [0]:
%sql
COPY INTO identifier(:catalog).bronze.bronze_customers
FROM '/Volumes/' :catalog '/bronze/raw/customers/'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true')

In [0]:
%sql
copy into identifier(:catalog).bronze.bronze_accounts
from "/Volumes/" :catalog "/bronze/raw/accounts/"
FILEFORMAT = csv
format_options ('header' = 'true', "inferSchema" = "true")
copy_options ('mergeSchema' = 'true')

In [0]:
%sql
copy into identifier(:catalog).bronze.bronze_branches
from "/Volumes/" :catalog "/bronze/raw/branches/"
FILEFORMAT = csv
format_options ('header' = 'true', "inferSchema" = "true")
copy_options ('mergeSchema' = 'true')

In [0]:
%sql
copy into identifier(:catalog).bronze.bronze_transactions
from "/Volumes/" :catalog "/bronze/raw/transactions/"
FILEFORMAT = csv
format_options ('header' = 'true', "inferSchema" = "true")
copy_options ('mergeSchema' = 'true')

# Data types and Schemas of all delta tables

In [0]:
df_customers = spark.table("dev.bronze.bronze_customers")
df_accounts = spark.table("dev.bronze.bronze_accounts")
df_branches = spark.table("dev.bronze.bronze_branches")
df_transactions = spark.table("dev.bronze.bronze_transactions")

In [0]:
df_customers.printSchema()

In [0]:
df_accounts.printSchema()

In [0]:
df_branches.printSchema()

In [0]:
df_transactions.printSchema()

# Add meta deta into delta tables

In [0]:
from pyspark.sql.functions import col

def add_file_metadata_to_table(source_path, target_table):
    df = (spark.read.format("csv").option("header", "true").option("inferSchema", "true")
          .load(source_path)
          .withColumn("file_name", col("_metadata.file_name"))
          .withColumn("file_path", col("_metadata.file_path"))
    )
    
    spark.sql(f"TRUNCATE TABLE {target_table}")
    df.write.mode("append").option("mergeSchema", "true").saveAsTable(target_table)
    return df

df_customers = add_file_metadata_to_table(f"/Volumes/{catalog}/bronze/raw/customers/", f"{catalog}.bronze.bronze_customers")
df_accounts = add_file_metadata_to_table(f"/Volumes/{catalog}/bronze/raw/accounts/", f"{catalog}.bronze.bronze_accounts")
df_branches = add_file_metadata_to_table(f"/Volumes/{catalog}/bronze/raw/branches/", f"{catalog}.bronze.bronze_branches")
df_transactions = add_file_metadata_to_table(f"/Volumes/{catalog}/bronze/raw/transactions/", f"{catalog}.bronze.bronze_transactions")